In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/Users/risshabsrinivas/Documents/IITM/DL_GENAI_PROJECT/dl-genai-project-26-t2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────
!pip install transformers torch tqdm -q

In [ ]:
# ── Cell 2: Config ────────────────────────────────────────────
import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# After adding dataset to notebook, it mounts here:
MODEL_PATH  = "/kaggle/input/mcq-solver-dummy-23f3002157"
TEST_PATH   = "/kaggle/input/YOUR_COMPETITION_SLUG/test.csv"
MAX_LENGTH  = 128
OPTIONS     = ["A", "B", "C", "D", "E"]
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Model path exists: {os.path.exists(MODEL_PATH)}")

In [ ]:
# ── Cell 3: Load model ────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE)
model.eval()
print("Model loaded.")

In [ ]:
# ── Cell 4: Load test data ────────────────────────────────────
test_df = pd.read_csv(TEST_PATH)
id_col = "id" if "id" in test_df.columns else "ID"
print(f"Test samples: {len(test_df)}")
test_df.head()

In [ ]:
# # ── Cell 5: Inference ─────────────────────────────────────────
# def predict_top3(row):
#     question = row.get("question", row.get("prompt", ""))
#     scores = []
#     for opt in OPTIONS:
#         text = str(row.get(opt, row.get(f"option_{opt.lower()}", "")))
#         pair = f"{question} [SEP] {opt}: {text}"
#         enc = tokenizer(
#             pair,
#             max_length=MAX_LENGTH,
#             padding="max_length",
#             truncation=True,
#             return_tensors="pt"
#         ).to(DEVICE)
#         with torch.no_grad():
#             logit = model(**enc).logits
#         score = torch.softmax(logit, dim=-1)[0][1].item()
#         scores.append(score)

#     ranked = sorted(range(5), key=lambda i: scores[i], reverse=True)
#     return " ".join([OPTIONS[i] for i in ranked[:3]])

# results = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
#     results.append({
#         "ID": row[id_col],
#         "Prediction": predict_top3(row)
#     })

# submission = pd.DataFrame(results)
# print(submission.head(10))

In [ ]:
# # ── Cell 6: Save submission ───────────────────────────────────
# submission.to_csv("submission.csv", index=False)
# print(f"submission.csv saved — {len(submission)} rows")

In [ ]:
import pickle
import torch
import numpy as np
import pandas as pd
from lstm_model import LSTMMCQModel

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DIR    = "/kaggle/input/mcq-lstm-23f3002157"
OPTION_COLS  = ["A", "B", "C", "D", "E"]
IDX_TO_LABEL = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}
MAX_LENGTH   = 128

In [ ]:
# ── Cell 2: Load vocab + model ────────────────────────────────
with open(f"{MODEL_DIR}/vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

model = LSTMMCQModel(
    vocab_size  = len(vocab),
    embed_dim   = 128,
    hidden_dim  = 256,
    num_layers  = 2,
    dropout     = 0.3,
    pad_idx     = 0,
).to(DEVICE)

model.load_state_dict(
    torch.load(f"{MODEL_DIR}/best_model.pt", map_location=DEVICE)
)
model.eval()
print("LSTM Loaded")

In [ ]:
# ── Cell 3: Inference ─────────────────────────────────────────
def encode_row(row):
    prompt = str(row["prompt"])
    ids = []
    for col in OPTION_COLS:
        text    = prompt + " " + str(row[col])
        tokens  = text.lower().translate(
            str.maketrans("", "", __import__("string").punctuation)
        ).split()[:MAX_LENGTH]
        encoded = [vocab.word2idx.get(t, 1) for t in tokens]
        encoded += [0] * (MAX_LENGTH - len(encoded))
        ids.append(encoded)
    return torch.tensor([ids], dtype=torch.long)

test_df = pd.read_csv("/kaggle/input/YOUR-COMPETITION-SLUG/test.csv")
results = []

for _, row in test_df.iterrows():
    input_ids = encode_row(row).to(DEVICE)
    with torch.no_grad():
        logits = model(input_ids)[0].cpu().numpy()
    top3 = " ".join([IDX_TO_LABEL[i] for i in np.argsort(logits)[::-1][:3]])
    results.append({"id": row["id"], "Prediction": top3})

submission = pd.DataFrame(results)
submission.to_csv("submission.csv", index=False)
print(f"Done. {len(submission)} rows.")
submission.head()